# Reproducing McGuire, Telian, Laboy-Juarez et al. (2016) from CRCNS ssc-4: behaviour and single units

Paper: McGuire LM, Telian G, Laboy-Juarez KJ, Miyashita T, Lee DJ, Smith KA, Feldman DE (2016),
"Short Time-Scale Sensory Coding in S1 during Discrimination of Whisker Vibrotactile Sequences",
PLoS Biology 14(8) e1002549, doi:10.1371/journal.pbio.1002549.

Dataset: CRCNS ssc-4 (doi:10.6080/K06D5QXN), the single file `ssc-4/live/AllRats.mat`, streamed
from the public bucket `crcnsarchive`. Version: CRCNS data sets are not versioned, so the version
is recorded as `unversioned` and the file is pinned by the MD5 sum in the data set's
`checksums.md5`, a929b803ca0feab5270e807f571d13eb (printed below from the bucket). There is only
this one version of the data set in the bucket (`live/`, no `old/`), so it is the one pinned; the
paper cites the data set by accession only. Outside inputs: the paper's supporting information
S1 Data (doi:10.1371/journal.pbio.1002549.s001) and S2 Data (doi:10.1371/journal.pbio.1002549.s002),
read for the three claims whose data are there (`data: elsewhere`) and as variants, and the
paper's S1 Table (doi:10.1371/journal.pbio.1002549.s010), set beside the baseline rates as a variant.

This is the second run of the notebook, revised after the verifier's first report: the claims
that depend on temporal modulation now use the data set's own modulated units, with the set
recomputed by the Methods' test as a variant (see the note after temporal-modulation-count).

The claims file (`claims/mcguire-2016.yaml`) was written from the version of record (Europe PMC,
PMC5004814) with the Methods and the supporting legends.

This notebook covers the behavioural and single-unit claims. The decoders (Figure 9) are in
`decoders.ipynb`, which uses the same helper (`ssc4.py`).

| claim | here |
|---|---|
| fig2ab-fms-smf-not-discriminated | tested (5 of 8 rats in the data) |
| training-days | tested (5 of 8 rats) |
| fig2c-choice-follows-mean-speed | tested (5 of 8 rats) |
| fig2d-fixed-panel-impairs | tested on S2 Data (elsewhere), after a reader's answer |
| dprime-fff-sss-range | tested (5 of 8 rats) |
| fig3-fsfs-sffs-intensity-only | tested on S1 Data (elsewhere) |
| units-recorded | tested |
| baseline-rates | tested |
| temporal-modulation-count | tested |
| modulated-higher-baseline | tested |
| fig4d-evoked-modulation-by-layer | tested |
| latency-by-layer | tested |
| fig4d-gradual-decrease | tested |
| fig6-regression-significant | tested |
| fig6e-unit-classes | tested |
| fig6fg-laminar-classes | tested |
| fig7c-fast-impulse-velocity | tested |
| fig7c-fast-sequence-rate | tested |
| fig7-fff-adaptation | tested |
| fig7-sss-no-adaptation | tested |
| s4-fms-middle-pulse | tested |
| fig7de-medium-suppressed | tested |
| fig8a-slow-positive-no-sequence-difference | tested |
| fig8b-slow-negative-no-impulse-response | tested |
| fig8d-slow-positive-right-choice-l5 | tested |
| fig8d-fast-medium-no-choice-bias | tested |
| fig8e-choice-divergence-125ms | tested |
| s5-slow-positive-no-integration | tested |
| s6-head-position-choice | tested on S2 Data (elsewhere) |
| s7-no-fs-l5-slow-positive | tested |
| fig9b, fig9c, fig9d, fig9e, fig9f, fig9g (seven claims) | in `decoders.ipynb` |

In [ ]:
import collections, warnings
import numpy as np, pandas as pd, fsspec
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency
import ssc4
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print(fsspec.open("https://crcnsarchive.s3-us-west-2.amazonaws.com/ssc-4/live/checksums.md5").open().read().decode())
D = ssc4.Data(ssc4.load_mat())
T, U = D.trials, D.units
S1 = ssc4.fetch_xlsx(ssc4.S1_DATA)
S2 = ssc4.fetch_xlsx(ssc4.S2_DATA)
print("S1 Data sheets", list(S1), "S2 Data sheets", list(S2))

## Data check

In [ ]:
print("sessions", len(D.A), "rats", T.rat.nunique())
print(T.groupby("rat").agg(sessions=("session", "nunique"), days=("date", "nunique"), trials=("trial", "size")))
print("\ntrials by sequence (StimNumber) and outcome")
T["outcome"] = np.select([T.complete, T.withdrew, T.timeout], ["choice", "early withdrawal", "no choice"], "other")
print(pd.crosstab(T.stim, T.outcome, margins=True))
print("\nStimName irregular names, by the StimNumber they carry (the descriptor says to use StimNumber)")
irr = T[~T.stim_name.isin(ssc4.STIMS)]
print(pd.crosstab(irr.stim_name, irr.stim))
print("\nchosen port codes", T.chosen.value_counts().to_dict(), " target codes", T.target.value_counts().to_dict())
print("completed trials with nose poke <= 250 ms:", int((T.complete & (T.np_dur <= 250)).sum()))
print("nose poke duration (ms), completed trials: min %.0f, median %.0f, max %.0f" % tuple(T[T.complete].np_dur.quantile([0, .5, 1])))
print("stimulus onset delay (ms): min %.0f max %.0f (Methods: 75-100)" % (T.delay.min(), T.delay.max()))
print("required nose poke (ms)", T.np_req.value_counts().to_dict())
print("\nunits", len(U))
print(pd.crosstab(U.layer, U.type, margins=True))
print("depth range (um) %.0f to %.0f" % (U.depth.min(), U.depth.max()))
print("electrodes per session", collections.Counter(len(np.atleast_1d(e.HistDepth)) for e in D.A))
print("distinct depths across electrodes within a session:", max(len(set(np.atleast_1d(e.HistDepth).round(3))) for e in D.A))
print("\nimpulse onset sets (ms) across sessions, StimTimes[:,0] by sequence FMS SMF FFF SSS:")
sets = collections.Counter(str([D.stimtimes[s][k][:, 0].tolist() for k in range(4)]) for s in range(len(D.A)))
for k, v in sets.items():
    ss = [s for s in range(len(D.A)) if str([D.stimtimes[s][j][:, 0].tolist() for j in range(4)]) == k]
    print(v, "sessions", k, "rats", sorted(set(T[T.session.isin(ss)].rat)))
print("\nfields of a session:", D.A[0]._fieldnames)
print("MapPosition (column of each electrode, named in the descriptor) present:", "MapPosition" in D.A[0]._fieldnames)
print("deposited labels", U.dep_label.value_counts().to_dict())
print("deposited TemporalPval < 0.05:", int((U.dep_temp_p < 0.05).sum()))
print("TempSig.Params:", {k: getattr(D.A[0].TempSig.Params, k) for k in D.A[0].TempSig.Params._fieldnames})
spk_all = np.concatenate([M.sum(axis=(0, 2)) for M in D.spk])
print("spikes per unit in the -100..400 ms span: min %d, median %d" % (spk_all.min(), np.median(spk_all)))

### Checks of the description against the values

1. The regression inputs deposited with each unit (`StimReg(...).Yval`) against spike counts from
   `TrialCells`, to find which trials the authors analysed.
2. The regressor (`Xval`) is the same for every unit and session.
3. The deposited regression R2 and p against a fresh OLS fit.
4. Recordings deposited twice: identical trial tables or spike trains across sessions.
5. Columns: the descriptor says to drop units in columns A1 and B2 using `MapPosition`.

In [ ]:
sel_names = {"all trials": lambda t: np.ones(len(t), bool), "nose poke > 250 ms": lambda t: (t.np_dur > 250).values,
             "completed": lambda t: t.complete.values}
match = collections.Counter()
for uid in U.uid:
    r = U.iloc[uid]
    Y = np.asarray(np.atleast_2d(D.A[r.session].StimReg)[r.unit][0].Yval).reshape(4, 36)
    for name, f in sel_names.items():
        M, t, _ = D.unit_trials(uid, f)
        C = np.stack([M[(t.stim == s).values, -ssc4.T0:180 - ssc4.T0].reshape(-1, 36, 5).sum((0, 2)) for s in ssc4.STIMS])
        match[name] += int(np.array_equal(C, Y))
print("units whose deposited Yval equals the summed counts, by trial selection:", dict(match), "of", len(U))

tset = {si: str([D.stimtimes[si][k][:, 0].tolist() for k in range(4)]) for si in range(len(D.A))}
X0 = {}
dx = collections.defaultdict(float)
for r in U.itertuples():
    x = np.array([np.asarray(np.atleast_2d(D.A[r.session].StimReg)[r.unit][w].Xval)[:, 1] for w in range(36)])
    k = tset[r.session]
    X0.setdefault(k, x)
    dx[k] = max(dx[k], np.abs(x - X0[k]).max())
print("largest difference of Xval between units, within each impulse-timing set:", [round(v, 6) for v in dx.values()])
print("between timing sets, largest difference:", round(float(np.abs(np.subtract(*list(X0.values())[:2])).max()), 1))

R2, P, B = ssc4.regress_all(D)
dep = np.array([[np.atleast_2d(D.A[r.session].StimReg)[r.unit][w].Regress.Stats[[0, 2]] for w in range(36)] for r in U.itertuples()])
print("units with no spikes in 0-180 ms (regression undefined):", int(np.isnan(R2).all(1).sum()))
print("max |R2 - deposited R2| = %.2e, max |log10 p - deposited| = %.2e" % (np.nanmax(np.abs(R2 - dep[..., 0])),
      np.nanmax(np.abs(np.log10(P + 1e-300) - np.log10(dep[..., 1] + 1e-300)))))
R2 = np.nan_to_num(R2, nan=0.0); P = np.nan_to_num(P, nan=1.0); B = np.nan_to_num(B)
dep_best = U.dep_peakbin.values
same = (R2.argmax(1) + 1) * 5 == dep_best
print("best window (highest R2) equals deposited PeakBin for", int(same.sum()), "of", len(U), "units; the others:",
      U.loc[~same, ["session", "unit", "dep_label", "dep_peakbin"]].to_dict("records"))

fp = T.groupby("session").apply(lambda t: hash((tuple(t.np_dur), tuple(t.stim), tuple(t.delay))))
print("sessions sharing a trial table with another:", int(fp.duplicated(keep=False).sum()))
sp = {}
for r in U.itertuples():
    M = D.spk[r.session][:, r.unit, :]
    sp.setdefault(hash(M[:20].tobytes()), []).append((r.session, r.unit))
print("unit pairs with identical spike matrices over the first 20 trials:", [v for v in sp.values() if len(v) > 1])

What the checks show (numbers above):

- The stored regression inputs equal the spike counts summed over completed trials for all 506
  units (204 units for nose poke > 250 ms, 1 for all trials). The authors' unit analyses used
  completed trials, and so does this notebook.
- The stored regressor is identical across units within two of the three impulse-timing sets and
  differs within the third; each unit's own stored regressor is used. A fresh fit reproduces the
  stored R2 and p to numerical precision; one unit has no spikes from 0 to 180 ms (stored label
  ImproperPeakBin).
- No two sessions share a trial table and no two units share spikes: no recording is deposited
  twice. The paper says a new set of units was sampled in every session (Methods, Neural
  Recordings), so units are counted once per session, an assumption the data cannot check.
- The data set holds 506 units (330 single, 176 multi) where the paper reports 473 (306, 167).
  The descriptor says units in columns A1 and B2 must be dropped using `MapPosition`, which the
  file does not contain; leaving out the fifth electrode of 18FB and 18Ri gives 459, not 473.
  The extra units cannot be identified.
- The stored temporal-modulation p-values were computed with a -50 to 300 ms window
  (`TempSig.Params`), where the Methods say 0 to 180 ms after nose poke entry.
- The impulse onsets differ between sessions: in 32 sessions (18FB, 18Ri, 18Ro) the second
  impulse starts at 50.25 or 60.25 ms, in 48 (all rats but 18FB) at 47.25 or 57.25 ms; each
  session's own onsets are used.
- Withdrawal times (nose poke duration minus onset delay) and the descriptor's trial fields are
  consistent with the Methods: delays 75 to 99 ms, completed trials all held over 250 ms.

### Clock of the spikes against the trial table

Spike times are said to be in ms from the first impulse onset. The second impulse starts about
50 ms after the first in FFF and FMS (the F-first sequences) and about 60 ms after it in SMF and
SSS. If the spikes share the trial clock, the population rate 55 to 60 ms after onset (an early
response to impulse 2 at 50 ms) is higher in FFF and FMS trials than in SMF and SSS trials, and a
window before stimulus onset shows no such difference.

In [ ]:
def group_contrast(a, b):
    out = []
    for si, M in enumerate(D.spk):
        t = T[T.session == si]
        m = t.complete.values
        st = t.stim.values[m]
        r = M[m][:, :, a - ssc4.T0:b - ssc4.T0].sum((1, 2)) * 1000 / (b - a)
        out.append(r[np.isin(st, ["FFF", "FMS"])].mean() - r[np.isin(st, ["SMF", "SSS"])].mean())
    return np.array(out)
for a, b in [(55, 60), (-60, -55)]:
    d = group_contrast(a, b)
    print("window %d to %d ms: sessions with F-first > S-first rate %d of %d; mean difference %.1f Hz (summed over units); sign test p = %.2g"
          % (a, b, (d > 0).sum(), len(d), d.mean(), stats.binomtest(int((d > 0).sum()), len(d)).pvalue))
for rat in sorted(T.rat.unique()):
    ss = sorted(T[T.rat == rat].session.unique())
    d = group_contrast(55, 60)[ss]
    print("  %s: %d of %d sessions F-first higher" % (rat, (d > 0).sum(), len(d)))

The response to impulse 2 is earlier in F-first sequences in 61 of 80 sessions (sign test
p = 2.7e-06), and a window before onset shows no difference (35 of 80, p = 0.31): consistent with
spikes and trial table sharing one clock. By rat this holds in 10 to 17 sessions of 12 to 22, and
in 18Ro in 11 of 19, where the check is weak.

## Behaviour (Figure 2)

Only the five recorded rats are in the data set (18FB, 18Ri, 18Ro, 62BS, 107St); the paper's
behaviour is from eight rats, and 62SC, 107Si and 62SS are not here. A trial enters when the rat
chose a port. The two trial readings of fig2ab-fms-smf-not-discriminated are (i) every trial with
a choice and (ii) completed trials with no premature withdrawal and nose poke over 250 ms.

In [ ]:
def behav_trials(reading):
    t = T[T.chosen >= 0]
    if reading == "ii":
        t = t[t.complete & ~t.withdrew & (t.np_dur > 250)]
    return t
for rd in ["i", "ii"]:
    print("reading", rd, "trials", len(behav_trials(rd)))
bt = behav_trials("i")
frac = bt.groupby(["rat", "stim"]).chosen.mean().unstack()[["FFF", "FMS", "SMF", "SSS"]]
ntr = bt.groupby(["rat", "stim"]).chosen.size().unstack()[["FFF", "FMS", "SMF", "SSS"]]
print("\nfraction of right choices, all sessions pooled (reading i)\n", frac.round(3))
print("\ntrials\n", ntr)

### fig2ab-fms-smf-not-discriminated

> All eight rats discriminated FFF from SSS but treated FMS and SMF as equivalent and
> intermediate; seven of eight showed no difference between FMS and SMF, and one rat (62SC) made
> more right choices to FMS than to SMF.

In [ ]:
rows = []
for rd in ["i", "ii"]:
    bt = behav_trials(rd)
    for rat, t in bt.groupby("rat"):
        f = t.groupby("stim").chosen.agg(["sum", "size"])
        p = f["sum"] / f["size"]
        cnt, nob = f.loc[["FMS", "SMF"], "sum"].values, f.loc[["FMS", "SMF"], "size"].values
        z, pz = proportions_ztest(cnt, nob)
        tab = np.array([cnt, nob - cnt])
        chi_c = chi2_contingency(tab, correction=True)[1]
        rows.append(dict(reading=rd, rat=rat, FFF=p.FFF, FMS=p.FMS, SMF=p.SMF, SSS=p.SSS,
                         FFF_gt_SSS=p.FFF > p.SSS,
                         between=(min(p.FMS, p.SMF) > p.SSS) and (max(p.FMS, p.SMF) < p.FFF),
                         FMS_minus_SMF=p.FMS - p.SMF, p_z=pz, p_chi2_corrected=chi_c,
                         sig_z=pz < 0.00625, sig_chi=chi_c < 0.00625))
fig2ab = pd.DataFrame(rows)
print(fig2ab.round(4).to_string())
print("\n62SC is not in the data set:", "62SC" not in set(T.rat))
print("\nVariant: the paper's own per-rat means (S1 Data, Figure 2B, mean over sessions), against the same from the data set")
f2 = S1["Fig 2"]
paper2b = pd.DataFrame(f2.iloc[23:31, 1:5].values.astype(float), index=f2.iloc[23:31, 0], columns=["FFF", "FMS", "SMF", "SSS"])
bt = behav_trials("i")
sess = bt.groupby(["rat", "session", "stim"]).chosen.mean().unstack().groupby("rat").mean()[["FFF", "FMS", "SMF", "SSS"]]
name = {"18FB": "18FB", "18Ri": "18Ri", "18Ro": "18Ro", "62BS": "62BS", "107St": "107ST"}
cmp = pd.concat({"data set (mean of sessions)": sess.rename(index=name), "S1 Data": paper2b.loc[[name[r] for r in sess.index]]}, axis=1)
print(cmp.round(3))
print("largest absolute difference: %.3f" % np.abs(cmp["data set (mean of sessions)"].values - cmp["S1 Data"].values).max())
print("\nS1 Data, all eight rats: FFF > SSS for", int((paper2b.FFF > paper2b.SSS).sum()), "of 8; FMS and SMF between for",
      int(((paper2b[["FMS", "SMF"]].min(axis=1) > paper2b.SSS) & (paper2b[["FMS", "SMF"]].max(axis=1) < paper2b.FFF)).sum()), "of 8")

**Result: partially reproduced.** In all five rats the data hold, FFF draws more right choices than SSS and FMS and SMF lie between them, and FMS against SMF is not significant at 0.00625 in any of them (smallest p 0.10 without and 0.11 with continuity correction, 62BS), under both trial readings, which select the same 14186 trials. Rat 62SC, the one the paper finds discriminating FMS from SMF, is not in the data set, nor are two other behaviour-only rats, so the claim is tested on 5 of 8 rats. The data set's per-rat means of sessions equal the S1 Data values for four rats and differ by up to 0.026 for 18FB.

### training-days

> Rats trained on FFF-FMS-SMF-SSS discrimination for 14.2 days on average (s.d. 4.4, range 8 to 22).

In [ ]:
days = T.groupby("rat").date.nunique()
sess = T.groupby("rat").session.nunique()
print(pd.DataFrame(dict(distinct_dates=days, sessions=sess)).to_string())
for nm, d in [("distinct recording dates", days), ("sessions (the paper's '12-22 d of recording' matches these)", sess)]:
    print("%s: mean %.1f, s.d. %.1f, min %d, max %d (n = %d rats)" % (nm, d.mean(), d.std(), d.min(), d.max(), len(d)))
print("first and last recording date per rat:")
print(T.groupby("rat").date.agg(["min", "max"]))


**Result: partially reproduced.** Counting distinct recording dates, the five rats in the data set have 11 to 20 days, mean 14.8 (s.d. 4.3): the mean and the maximum are within tolerance of 14.2 and 22, the minimum (11) is not within 0.8 of 8. Counting sessions, which match the paper's "12-22 d of recording", gives 12 to 22, mean 16.0: the minimum is still outside. The data hold only the recording days of five of the eight rats.

### fig2c-choice-follows-mean-speed

> Relative right-port choice (fraction of right choices for a stimulus minus the mean over all
> stimuli) was strongly related to the mean speed of the sequence for all rats.

In [ ]:
speed = pd.Series(ssc4.MEAN_SPEED)[["FFF", "FMS", "SMF", "SSS"]]
def fig2c_table(fr):
    rel = fr.sub(fr.mean(axis=1), axis=0)
    out = pd.DataFrame(dict(
        order_a=(rel.FFF > rel[["FMS", "SMF"]].max(axis=1)) & (rel[["FMS", "SMF"]].min(axis=1) > rel.SSS),
        r=[stats.pearsonr(speed.values, rel.loc[i].values)[0] for i in rel.index]))
    out["r_gt_0.8"] = out.r > 0.8
    return pd.concat([rel.round(3), out], axis=1)
for rd in ["i", "ii"]:
    fr = behav_trials(rd).groupby(["rat", "stim"]).chosen.mean().unstack()[["FFF", "FMS", "SMF", "SSS"]]
    print("reading", rd); print(fig2c_table(fr).round(3))
print("\nVariant, all eight rats from S1 Data Figure 2B means:")
print(fig2c_table(paper2b).round(3))

**Result: reproduced.** For each of the five rats relative right choice orders FFF > FMS and SMF > SSS, and its Pearson correlation with mean speed is 0.967 to 0.998, under both trial readings. The S1 Data means of all eight rats give the same (r 0.967 to 0.995).

### fig2d-fixed-panel-impairs (S2 Data, S2B Fig; S1 Data, Figure 2D)

> Fixing the panel in place, with the piezo moving behind it, impaired FFF against SSS
> discrimination in all but one of six rats tested.

The data set holds only the four normal sequences (StimNumber 1 to 4, table above); the
fixed-panel control was behaviour only and is not deposited. Following the reader's answer
(questions/mcguire-2016.yaml), the claim is tested on the per-rat values of S2B Fig in S2 Data:
FFF minus SSS right-choice fraction on normal (moving-panel) and fixed-panel trials, paired t
test across rats, two-sided (S2B Fig legend). The rat at 0 enters under reading (ii) and is left
out under reading (i). The d-prime of Figure 2D, from S1 Data, is a variant. A last check asks
which normal sessions the moving-panel values summarise, by setting them beside the S1 Data
Figure 2B means and the data set's sessions.

In [ ]:
def paired(d, a, b):
    dd = d[b] - d[a]
    r = stats.ttest_rel(d[a], d[b])
    ci = stats.t.interval(0.95, len(dd) - 1, dd.mean(), stats.sem(dd))
    return "n = %d, fixed lower in %d of %d, mean change %.3f (95%% CI %.3f to %.3f), paired t = %.2f, p = %.4f" % (
        len(d), (d[b] < d[a]).sum(), len(d), dd.mean(), *ci, r.statistic, r.pvalue)

s2b = S2["Fig S2"].iloc[38:41, 0:9]
s2b = pd.DataFrame(s2b.iloc[1:, 1:].values.astype(float), index=s2b.iloc[1:, 0].values, columns=s2b.iloc[0, 1:].values).T
s2b["fixed_minus_moving"] = s2b.FixedPanel - s2b.MovingPanel
print("S2 Data, S2B Fig: FFF minus SSS right-choice fraction per rat (range %.3f to %.3f)" % (s2b.iloc[:, :2].min().min(), s2b.iloc[:, :2].max().max()))
print(s2b.round(4))
both = s2b.dropna()
print("rats with a fixed-panel value:", len(both))
for nm, d in [("(ii) rat at 0 kept", both), ("(i) rat at 0 left out", both[both.FixedPanel > 0])]:
    print("%s: %s" % (nm, paired(d, "MovingPanel", "FixedPanel")))

f2 = S1["Fig 2"]
dp = pd.DataFrame(f2.iloc[45:47, 2:10].values.astype(float), index=["normal", "fixed"], columns=f2.iloc[44, 2:10].values).T
print("\nVariant: S1 Data, Figure 2D d-prime per rat")
print(dp)
db = dp.dropna()
for nm, d in [("(ii) rat at 0 kept", db), ("(i) rat at 0 left out", db[db.fixed > 0])]:
    print("%s: %s" % (nm, paired(d, "normal", "fixed")))

p2b = pd.DataFrame(f2.iloc[23:31, [1, 4]].values.astype(float), index=f2.iloc[23:31, 0].str.upper(), columns=["FFF", "SSS"])
ses = T[T.chosen >= 0].groupby(["rat", "session", "stim"]).chosen.mean().unstack()
ses = (ses.FFF - ses.SSS).rename("d").reset_index()
cmp = pd.DataFrame({"S2B moving panel": s2b.MovingPanel.rename(index=str.upper), "S1 2B means, FFF-SSS": p2b.FFF - p2b.SSS})
cmp["data set, mean of sessions"] = ses.groupby("rat").d.mean().rename(index=str.upper)
cmp["data set, last session"] = ses.sort_values("session").groupby("rat").d.last().rename(index=str.upper)
print("\nWhich normal sessions the moving-panel values summarise")
print(cmp.round(4))
a = f2.iloc[4:17, 1:5].values.astype(float); fx = f2.iloc[4:8, 6].values.astype(float)
print("\nS1 Data, Figure 2A sheet (labelled 'Rat 107ST'): mean FFF over its 13 days %.4f; fixed-panel FFF minus SSS %.4f, d-prime %.3f"
      % (a[:, 0].mean(), fx[0] - fx[3], stats.norm.ppf(fx[0]) - stats.norm.ppf(fx[3])))
print("S1 Data, Figure 2B mean FFF fraction: 107ST %.4f, 107Si %.4f" % (paper2b.loc["107ST", "FFF"], paper2b.loc["107Si", "FFF"]))

**Result: reproduced.** In S2 Data, fixed-panel discrimination is lower than moving-panel discrimination in all six rats tested, by 0.067 (18Ri) to 0.374 (107SI); the paired t test gives p = 0.0120 with the rat at 0 kept (reading ii, the paper's p = 0.012) and p = 0.0308 without it (reading i), mean change -0.169 (95% CI -0.282 to -0.056) and -0.174 (-0.322 to -0.026). Five or more of six lower and p < 0.05 hold under both readings. The d-prime of Figure 2D from S1 Data is also lower in all six rats (p = 0.0051 and 0.0176). The moving-panel values of S2B equal the Figure 2B means for the two rats without the control and differ from them, and from the data set's mean and last session, for the six with it, so they summarise sessions this data set does not hold. The S1 Data Figure 2A sheet, labelled 107ST, has a mean FFF fraction (0.7485) and fixed-panel values (0.0991, d-prime 0.277) that are those of 107Si in Figure 2B, S2B and Figure 2D.

### dprime-fff-sss-range

> Discrimination of FFF against SSS in normal sessions was modest, with d-prime from 0.5 to 1.5 across rats.

In [ ]:
def dprime(hr, fa, n1, n2):
    hr = np.clip(hr, 0.5 / n1, 1 - 0.5 / n1)
    fa = np.clip(fa, 0.5 / n2, 1 - 0.5 / n2)
    return stats.norm.ppf(hr) - stats.norm.ppf(fa)
for rd in ["i", "ii"]:
    bt = behav_trials(rd)
    pooled, persess = {}, {}
    for rat, t in bt.groupby("rat"):
        f = t.groupby("stim").chosen.agg(["mean", "size"])
        pooled[rat] = dprime(f.loc["FFF", "mean"], f.loc["SSS", "mean"], f.loc["FFF", "size"], f.loc["SSS", "size"])
        v = []
        for s, ts in t.groupby("session"):
            g = ts.groupby("stim").chosen.agg(["mean", "size"])
            if {"FFF", "SSS"} <= set(g.index):
                v.append(dprime(g.loc["FFF", "mean"], g.loc["SSS", "mean"], g.loc["FFF", "size"], g.loc["SSS", "size"]))
        persess[rat] = np.mean(v)
    dp = pd.DataFrame(dict(pooled=pooled, session_mean=persess))
    print("reading", rd); print(dp.round(3))
    print("  pooled: range %.2f to %.2f; session mean: range %.2f to %.2f" % (dp.pooled.min(), dp.pooled.max(), dp.session_mean.min(), dp.session_mean.max()))
dp8 = pd.Series(dprime(paper2b.FFF.values, paper2b.SSS.values, 1e9, 1e9), index=paper2b.index)
print("Variant: d-prime from the S1 Data means of all eight rats (mean of sessions, not pooled):", dp8.round(2).to_dict())

**Result: partially reproduced.** Per-rat d-prime over pooled sessions is 0.45 to 1.09 (mean of sessions 0.46 to 1.23): the lowest value is within tolerance of 0.5, the highest is below 1.35 under both readings. The three rats missing from the data set include the two with the highest d-prime by S1 Data (62SS 1.32, 107Si 1.17).

### fig3-fsfs-sffs-intensity-only (S1 Data, Figure 3)

> Two rats discriminated FSFS from SFFS when the sequences differed in mean speed (70 and 69.2%
> correct) but not when mean speed was the same (52 and 53%), even after more than 20 days;
> d-prime was 1.02 against 0.12.

In [ ]:
f3 = S1["Fig 3"]
print("columns:", f3.iloc[2, 2:6].tolist(), f3.iloc[3, 2:6].tolist())
vals = f3.iloc[4:55, 1:6].apply(pd.to_numeric, errors="coerce")
vals.columns = ["day", "58B same", "58B diff", "60W same", "60W diff"]
summ = pd.DataFrame({c: dict(sessions=vals[c].notna().sum(), mean_pct=100 * vals[c].mean(),
                             sem_pct=100 * vals[c].std() / np.sqrt(vals[c].notna().sum()), last_day=vals.day[vals[c].notna()].max())
                     for c in vals.columns[1:]}).T
print(summ.round(2))
dpr = f3.iloc[59:62, 0:5]
print(dpr.to_string())
dd = pd.to_numeric(dpr.iloc[1, [1, 3]]).values
ds = pd.to_numeric(dpr.iloc[2, [1, 3]]).values
print("d-prime different intensity: mean of rats %.3f (60W %.3f, 58B %.3f)" % (dd.mean(), dd[0], dd[1]))
print("d-prime same intensity: mean of rats %.3f (60W %.3f, 58B %.3f)" % (ds.mean(), ds[0], ds[1]))

**Result: reproduced.** From S1 Data, 58B is 69.98 +- 1.47% correct and 60W 69.17 +- 1.57% with different intensities, and 51.74 +- 0.81% and 53.02 +- 0.82% with the same intensity, over sessions up to day 56 and 52. The d-prime is 1.004 and 0.130 as the mean of the two rats (reading a), all within tolerance.

## Units

### units-recorded

> The recordings yielded 306 single units and 167 multi-unit clusters (473 units) over 80
> sessions in five rats, 3.8 single units per session on average (range 0 to 11).

In [ ]:
su = U.type.str.startswith("SU")
per = U[su].groupby("session").size().reindex(range(len(D.A)), fill_value=0)
print("single units %d (RS %d, FS %d), multi-unit clusters %d, total %d" % (su.sum(), (U.type == "SU_RS").sum(), (U.type == "SU_FS").sum(), (~su).sum(), len(U)))
print("sessions %d, rats %d" % (len(D.A), U.rat.nunique()))
print("single units per session: mean %.2f, min %d, max %d" % (per.mean(), per.min(), per.max()))
print(U.groupby(["rat", "type"]).size().unstack())
print("\nby electrode (18FB and 18Ri have a fifth electrode; leaving it out does not give 473):")
print(pd.crosstab(U.elect, U.type, margins=True))

**Result: reproduced.** The data set holds 330 single units and 176 multi-unit clusters over 80 sessions in five rats, 4.12 single units per session (0 to 12), each within tolerance. The total is 506 against the paper's 473; the 33 extra units cannot be identified, as the column map is missing.

### baseline-rates

> Mean firing rate in the 25-ms prestimulus baseline in the nose poke was 6 to 10 Hz across
> layers for RS units, 8 to 32 Hz for FS units, and higher for multi-unit clusters.

In [ ]:
U["base25"] = [ssc4.rate(D.unit_trials(u)[0], -25, 0).mean() for u in U.uid]
U["base25_all"] = [ssc4.rate(D.unit_trials(u, lambda t: np.ones(len(t), bool))[0], -25, 0).mean() for u in U.uid]
order = ["L2/3", "L4", "L5a", "L5b", "L6"]
bl = U.groupby(["layer", "type"]).base25.agg(["mean", "size"]).unstack().loc[order]
print(bl.round(2))
for ty in ["SU_RS", "SU_FS"]:
    m = bl[("mean", ty)]
    print("%s: lowest layer mean %.2f Hz (%s), highest %.2f Hz (%s)" % (ty, m.min(), m.idxmin(), m.max(), m.idxmax()))
mu = bl[("mean", "MU")]
print("MU higher than RS in %d of 5 layers; higher than RS and FS in %d of 5" % ((mu > bl[("mean", "SU_RS")]).sum(),
      ((mu > bl[("mean", "SU_RS")]) & (mu > bl[("mean", "SU_FS")])).sum()))
rows_s1t, md5_s1t = ssc4.fetch_docx_table(ssc4.S1_TABLE)
print("\nS1 Table of the paper (doi:10.1371/journal.pbio.1002549.s010, MD5 %s when read): baseline mean, units" % md5_s1t)
s1t = pd.DataFrame([dict(layer=r[0].replace("L23", "L2/3"), type={"RS": "SU_RS", "FS": "SU_FS", "MU": "MU"}[r[1]],
                         s1t_mean=float(r[2].split("±")[0]), s1t_n=int(r[8])) for r in rows_s1t if len(r) > 8 and r[1] in ("RS", "FS", "MU")])
s1t = s1t.set_index(["layer", "type"])
here = U.groupby(["layer", "type"]).base25.agg(["mean", "size"]).rename(columns={"mean": "here_mean", "size": "here_n"})
cmpt = s1t.join(here)
cmpt["n_extra_here"] = cmpt.here_n - cmpt.s1t_n
print(cmpt.round(2).to_string())
print("S1 Table units: %d; here: %d. RS layer means in S1 Table: %.1f to %.1f Hz; FS: %.1f to %.1f Hz"
      % (cmpt.s1t_n.sum(), cmpt.here_n.sum(), s1t.xs("SU_RS", level=1).s1t_mean.min(), s1t.xs("SU_RS", level=1).s1t_mean.max(),
         s1t.xs("SU_FS", level=1).s1t_mean.min(), s1t.xs("SU_FS", level=1).s1t_mean.max()))
print("\nVariant, all trials (not only completed):")
print(U.groupby(["layer", "type"]).base25_all.mean().unstack().loc[order].round(2))

**Result: partially reproduced.** RS single-unit layer means span 5.93 (L2/3) to 11.40 Hz (L5a), FS 7.59 to 30.18 Hz, and multi-units are higher than RS in 5 of 5 layers. The RS maximum is above the 11 Hz limit; the other three bounds are within tolerance. With all trials the RS maximum is 11.49 Hz. The paper's S1 Table gives 10.8 Hz for L5a RS units (6.0 to 10.8 Hz for RS, 7.9 to 31.6 Hz for FS) over 68 units where the data set has 77; the data set has 33 more units than S1 Table, all in L2/3 to L5b and none in L6, so the extra units are a likely source of the difference.

### temporal-modulation-count

> 305 of 473 units (63.5%) had firing rates significantly modulated in time during the nose poke period.

Readings of the window: (a) 0 to 180 ms after nose poke entry, (b) 0 to 180 ms after stimulus
onset. The two statistics the claim names give the same p-value under the uniform permutation
(helper docstring), so one is run. A third window, -50 to 300 ms after onset, is the one stored
with the data set's own p-values (`TempSig.Params`), shown as a variant.

In [ ]:
pv = {"a: 0-180 after NP entry": ssc4.temporal_pvals(D, "np", seed=1),
      "b: 0-180 after onset": ssc4.temporal_pvals(D, "stim", seed=2),
      "variant: -50-300 after onset": ssc4.temporal_pvals(D, "deposit", seed=3)}
depmod = U.dep_temp_p.values < 0.05
for k, p in pv.items():
    m = p < 0.05
    print("%-30s modulated %d of %d (%.1f%%); agrees with the deposited p < 0.05 on %d units (%.1f%%); Spearman r of p with the deposited p %.3f" %
          (k, m.sum(), len(m), 100 * m.mean(), (m == depmod).sum(), 100 * np.mean(m == depmod), stats.spearmanr(p, U.dep_temp_p, nan_policy='omit').statistic))
print("deposited TemporalPval < 0.05: %d of %d (%.1f%%)" % (depmod.sum(), len(U), 100 * depmod.mean()))
for thr in [0.01, 0.05, 0.1]:
    print("threshold %.2f: reading a %d, reading b %d" % (thr, (pv["a: 0-180 after NP entry"] < thr).sum(), (pv["b: 0-180 after onset"] < thr).sum()))
print("\nVariant null (asked by the verifier): each trial's 10-ms bins shuffled independently, 1000 permutations")
rng = np.random.default_rng(4)
for nm, rd in [("-50-300 after onset (stored window)", "deposit"), ("a: 0-180 after NP entry", "np")]:
    p = np.array([ssc4.perm_pval_shuffle(ssc4.temporal_bins(D, u, rd), rng) for u in U.uid])
    m = p < 0.05
    print("%-38s modulated %d of %d (%.1f%%); agrees with the deposited p < 0.05 on %.1f%% of units; Spearman r of p with the deposited p %.3f"
          % (nm, m.sum(), len(m), 100 * m.mean(), 100 * np.mean(m == depmod), stats.spearmanr(p, U.dep_temp_p, nan_policy='omit').statistic))
U["mod_a"] = pv["a: 0-180 after NP entry"] < 0.05
U["mod"] = depmod
print("\nby rat, reading a"); print(U.groupby("rat")["mod_a"].agg(["sum", "size"]))


**Result: reproduced.** With the Methods' window, 0 to 180 ms after nose poke entry (reading a), 310 of 506 units (61.3%) are modulated, within tolerance of 305 and 63.5%; with 0 to 180 ms after onset (reading b) 261 (51.6%). The data set's own p-values also mark 310, but they agree with reading (a) unit by unit for only 73.1% of units. Neither the stored window (-50 to 300 ms, 363 units, 81.6% agreement) nor a null that shuffles each trial's bins (365 units and 82.0% on the stored window, 321 and 73.7% on the Methods' window) gives the stored set, so the authors' test is not repeated exactly here.

The units used from here on are those the data set marks as temporally modulated
(`TempSig.TemporalPval < 0.05`, 310 units). The data descriptor names them as the units of the
paper's analyses: "For many analyses in the paper, we only analyzed units whose firing rate was
significantly temporally modulated with time during the trial. These units can be identified by:
AllRats(expt).TempSig.TemporalPval(unit) < 0.05". The set recomputed here with the Methods' test
(reading a) has the same count but is a different set of units (agreement printed above), so it is
not the authors' set. The first run of this attempt used the recomputed set; on the verifier's
report it was replaced by the deposited set, and the recomputed set is shown as a variant in each
claim that depends on it. This is an assumption: the paper's 305 of 473 units cannot be
identified exactly, as the 33 extra units of the data set cannot be told apart (units-recorded).


### modulated-higher-baseline

> Temporally modulated units were distributed uniformly across columns and layers and had higher
> baseline firing rates than non-modulated units.

In [ ]:
for nm, col in [("deposited modulated set (main)", "mod"), ("variant: recomputed set, reading a", "mod_a")]:
    rs = U[U.type == "SU_RS"]
    tab = rs.groupby(["layer", col]).base25.agg(["mean", "sem", "size"]).unstack().loc[order]
    print(nm); print(tab.round(2))
    hi = (tab[("mean", True)] > tab[("mean", False)])
    print("modulated RS mean higher in %d of 5 layers" % hi.sum())
    ct = pd.crosstab(U.layer, U[col]).loc[order]
    print(ct.assign(frac=ct[True] / ct.sum(axis=1)).round(3))
    print("chi-square layer x modulation (all units): p = %.3f\n" % chi2_contingency(ct.values)[1])
print("columns cannot be tested: the column of each electrode (MapPosition) is not in the data set")


**Result: reproduced.** On the deposited modulated set, modulated RS single units have higher baseline rates than non-modulated ones in 5 of 5 layers (in L5a 11.46 against 11.35 Hz, a small margin), and the same holds on the recomputed set (5 of 5). The fraction modulated is 0.49 to 0.73 across layers on the deposited set, chi-square p = 0.002, so on this variant the modulated units are not spread evenly over layers (p = 0.105 on the recomputed set). Columns cannot be tested without the column map.

### fig4d-evoked-modulation-by-layer

> The mean impulse-evoked firing rate modulation, above the pre-impulse baseline, was 14.2 Hz in
> L2/3, 15.2 in L4, 6.3 in L5a, 14.4 in L5b and 7.0 in L6, greatest in L2/3, L4 and L5b.

In [ ]:
modu = U.uid[U["mod"] | U["mod_a"]].values
ev = {}
for u in modu:
    segs = ssc4.impulse_segments(D, u)
    ev[u] = dict(pooled=ssc4.evoked_peak(segs, True), each=ssc4.evoked_peak(segs, False))
expect = pd.Series({"L2/3": 14.2, "L4": 15.2, "L5a": 6.3, "L5b": 14.4, "L6": 7.0})
tol = pd.Series({"L2/3": 2.3, "L4": 1.9, "L5a": 1.2, "L5b": 2.3, "L6": 1.5})
def fig4d_table(units):
    rows = []
    for u in units:
        for pool in ["pooled", "each"]:
            pk, mn, lat = ev[u][pool]
            rows.append(dict(uid=u, layer=U.layer[u], pool=pool, A_signed=pk, A_abs=abs(pk), B_mean=mn))
    EV = pd.DataFrame(rows)
    res = {}
    for pool in ["pooled", "each"]:
        for meas in ["A_signed", "A_abs", "B_mean"]:
            res[(meas, pool)] = EV[EV.pool == pool].groupby("layer")[meas].mean().loc[order]
    R = pd.DataFrame(res).T
    R["n_within"] = [(np.abs(R.loc[i, order] - expect) <= tol).sum() for i in R.index]
    R["within_all"] = R.n_within == 5
    R["L5a_L6_lowest"] = [set(R.loc[i, order].astype(float).nsmallest(2).index) == {"L5a", "L6"} for i in R.index]
    return R, EV
print("expected", expect.to_dict(), "tolerance", tol.to_dict())
for nm, col in [("deposited modulated set (main)", "mod"), ("variant: recomputed set, reading a", "mod_a")]:
    R, EV = fig4d_table(U.uid[U[col]].values)
    print("\n" + nm); print(R.round(2).to_string())
    print("units per layer", EV[EV.pool == "pooled"].groupby("layer").size().loc[order].to_dict())
    print("SEM across units, A_abs each:", EV[EV.pool == "each"].groupby("layer").A_abs.sem().loc[order].round(2).to_dict())


**Result: partially reproduced.** On the deposited modulated set no combination puts all five layers within tolerance; the closest, the signed peak over pooled impulses (A, a, i), has 4 of 5 (14.33, 14.23, 2.55, 13.11, 5.76 Hz, L5a outside). L5a and L6 are the two lowest layers under every combination, on both unit sets. The recomputed set gives at most 3 of 5.

### latency-by-layer

> Among impulse-responsive units, peak response latency was shortest in L4, L5a and L5b (9.8,
> 10.3 and 12.0 ms) and longest in L2/3 and L6 (13.8 and 16.1 ms).

In [ ]:
lt = []
for u in modu:
    segs = ssc4.impulse_segments(D, u)
    tt, p2, p1 = ssc4.impulse_ttest(segs)
    row = dict(uid=u, layer=U.layer[u], p_one=p1, p_two=p2)
    for sm in ["centred", "trailing", "forward", "none"]:
        row[sm] = ev[u]["pooled"][2] if sm == "centred" else ssc4.evoked_peak(segs, True, sm)[2]
    lt.append(row)
LT = pd.DataFrame(lt)
lexp = pd.Series({"L4": 9.8, "L5a": 10.3, "L5b": 12.0, "L2/3": 13.8, "L6": 16.1})
print("expected", lexp.loc[order].to_dict(), "tolerance 10%")
rows = []
for setname, col in [("deposited set", "mod"), ("recomputed set (a)", "mod_a")]:
    for test in ["p_one", "p_two"]:
        r = LT[LT.uid.isin(U.uid[U[col]]) & (LT[test] < 0.05)]
        print("%s, %s: impulse-responsive units per layer %s" % (setname, test, r.groupby("layer").size().loc[order].to_dict()))
        for sm in ["centred", "trailing", "forward", "none"]:
            for summ in ["mean", "median"]:
                v = r.groupby("layer")[sm].agg(summ).loc[order]
                within = np.abs(v - lexp.loc[order]) <= 0.1 * lexp.loc[order]
                rows.append(dict(set=setname, test=test, smoothing=sm, summary=summ, **v.round(2).to_dict(),
                                 n_within=int(within.sum()), ordered=bool(max(v[["L4", "L5a", "L5b"]]) < min(v[["L2/3", "L6"]]))))
LTS = pd.DataFrame(rows)
print(LTS.to_string())


**Result: partially reproduced.** With the centred 10-ms boxcar, latencies on the deposited set are 15 to 22 ms in every layer, longer than the paper's 9.8 to 16.1 ms, and no layer is within tolerance. The ordering (L4, L5a, L5b below L2/3 and L6) holds with the two-sided test (mean and median) and not with the one-sided test, where L5a is among the latest. Placing the boxcar forward (t to t+9 ms) shortens latencies by about 5 ms and puts at most 2 of 5 layers within tolerance on the deposited set; on that set the trailing boxcar puts none and the unsmoothed PSTH at most 2, and on the recomputed set the forward boxcar and the unsmoothed PSTH put at most 3. No reading puts all five within tolerance.

### fig4d-gradual-decrease

> Superimposed on the phasic impulse responses, average firing rate decreased gradually during
> the nose poke period in all layers except L5a.

In [ ]:
def unit_psth(u, a=-25, b=180, sel=None):
    M, t, _ = D.unit_trials(u, sel)
    return M[:, a - ssc4.T0:b - ssc4.T0].mean(0) * 1000
tax = np.arange(-25, 180)
for nm, col in [("deposited modulated set (main)", "mod"), ("variant: recomputed set, reading a", "mod_a")]:
    rows = []
    for lay in order:
        us = U[U[col] & (U.layer == lay)].uid
        pop = np.mean([unit_psth(u) for u in us], 0)
        slope = stats.linregress(tax, pop)
        rows.append(dict(layer=lay, n=len(us), slope_hz_per_100ms=100 * slope.slope, slope_p=slope.pvalue,
                         baseline=pop[:25].mean(), last30=pop[-30:].mean(), last30_minus_base=pop[-30:].mean() - pop[:25].mean()))
    GD = pd.DataFrame(rows).set_index("layer")
    GD["decrease_a"] = GD.slope_hz_per_100ms < 0
    GD["decrease_b"] = GD.last30_minus_base < 0
    print(nm); print(GD.round(3)); print()


**Result: not reproduced.** On the deposited modulated set the population rate decreases in every layer, L5a included: the slope from -25 to 180 ms is -0.94 Hz per 100 ms in L5a (p < 0.001) and -1.42 to -2.12 in the others, and the last 30 ms are below baseline in all five layers (by 1.70 Hz in L5a). The recomputed set gives the same (L5a -0.88 Hz per 100 ms). The claim requires no decrease in L5a.

## Regression and unit classes (Figure 6)

### fig6-regression-significant

> Of the 305 temporally modulated units, 204 had a significant regression of firing rate on
> integrated panel speed for at least one of the 36 integration windows.

In [ ]:
sig = (P < 0.05 / 36).any(1)
mod = U["mod"].values
mod_a = U["mod_a"].values
print("significant regression among the deposited modulated units (main): %d of %d" % ((sig & mod).sum(), mod.sum()))
print("variant, among units modulated by the recomputed test (reading a): %d of %d" % ((sig & mod_a).sum(), mod_a.sum()))
print("among all units: %d of %d" % (sig.sum(), len(sig)))
R2r, Pr, Br = ssc4.regress_all(D, use_rate=True)
sigr = (Pr < 0.05 / 36).any(1)
print("variant, rate per trial instead of summed counts: %d of %d modulated" % ((sigr & mod).sum(), mod.sum()))
print("by rat:"); print(pd.Series(sig & mod).groupby(U.rat).sum())


**Result: reproduced.** 207 of the 310 deposited modulated units have a significant regression at p < 0.05/36, within tolerance of 204; on the recomputed set, 194 of 310. The regression reproduces the data set's stored R2 and p to numerical precision.

### fig6e-unit-classes

> Units were classed by best-fit integration window as Fast (5 to 20 ms, 158 units, 52%, positive
> slopes), Medium (25 to 55 ms, 52 units, largely negative slopes), Slow Positive (51) and Slow
> Negative (44); 44% of units had very fast (5 to 10 ms) windows.

In [ ]:
exp6 = dict(Fast=158, Medium=52, SlowPos=51, SlowNeg=44)
def class_table(modmask):
    out, rows = {}, []
    for which in ["all", "sig"]:
        for m55 in [False, True]:
            lab, best, slope, sg = ssc4.unit_classes(R2, P, B, modmask, which, m55)
            key = "%s, 55 ms %s" % ("a all modulated" if which == "all" else "b significant only", "Medium" if m55 else "Slow")
            out[key] = lab
            c = collections.Counter(lab)
            fast, med = lab == "Fast", lab == "Medium"
            row = dict(reading=key, **{k: c[k] for k in exp6})
            row["pct_fast_of_mod"] = 100 * fast.sum() / modmask.sum()
            row["pct_5_10_of_mod"] = 100 * ((best <= 10) & (lab != "none")).sum() / modmask.sum()
            row["fast_pos_frac"] = np.mean(slope[fast] > 0)
            row["medium_neg_frac"] = np.mean(slope[med] < 0)
            row["counts_within"] = all(abs(c[k] - v) <= 0.1 * v for k, v in exp6.items())
            row["pcts_within"] = abs(row["pct_fast_of_mod"] - 52) <= 5.2 and abs(row["pct_5_10_of_mod"] - 44) <= 4.4
            rows.append(row)
    return out, pd.DataFrame(rows).set_index("reading")
cls, C6 = class_table(mod)
print("deposited modulated set (main)"); print(C6.round(3).to_string())
cls_a, C6a = class_table(mod_a)
print("\nvariant: recomputed modulated set (reading a)"); print(C6a.round(3).to_string())
depc = ssc4.deposited_classes(D)
print("\ndeposited labels (DFLabel; FastPos/FastNeg split into Fast and Medium at 20 ms by PeakBin):", collections.Counter(depc))
lab_main = cls["a all modulated, 55 ms Medium"]
print("reading (a, 55 ms Medium) on the deposited set against the deposited labels:")
print(pd.crosstab(pd.Series(lab_main, name="this attempt"), pd.Series(depc, name="deposited")))
print("units whose best window is 55 ms:", U.loc[(R2.argmax(1) + 1) * 5 == 55, ["session", "unit", "dep_label", "dep_peakbin"]].to_dict("records"))


**Result: reproduced.** Classing all deposited modulated units by their best window (reading a) gives 160 Fast, 52 Medium, 51 Slow Positive and 47 Slow Negative with 55 ms as Slow, and 160, 53, 51 and 46 with 55 ms as Medium; all counts are within tolerance under both. Fast units are 51.6% of modulated units and 43.5% have 5 to 10 ms windows, both within tolerance; 94% of Fast units have positive slopes and 69 to 70% of Medium units negative. Classing only units with a significant regression (reading b) gives 125, 30 or 31, 28, 24 or 23, outside tolerance. With 55 ms as Medium the classes equal the data set's labels unit for unit. On the recomputed modulated set Slow Negative is 50 or 51, above its limit of 48.4.

Downstream claims use the classes of reading (a) on the deposited modulated set, with 55 ms as
Medium: the paper's four counts sum to 305, all temporally modulated units, and the Results put
55 ms in Medium ("Units with 25–55 ms integration windows"), as do the data set's own labels for
the one unit whose best window is 55 ms. On this set these classes equal the deposited labels unit
for unit (table above). The first run used reading (a) with 55 ms Slow on the recomputed set; the
classes of reading (a), 55 ms Medium, on the recomputed set are the variant shown from here on.


### fig6fg-laminar-classes

> Fast units were 73% of temporally modulated units in L2/3 and L4, 50% in L5 and 23% in L6;
> slow units were 13% in L2/3 and L4, 31% in L5 and 56% in L6.

In [ ]:
grp = {"L2/3": "L2/3+L4", "L4": "L2/3+L4", "L5a": "L5", "L5b": "L5", "L6": "L6"}
expF = {"L2/3+L4": 73, "L5": 50, "L6": 23}
expS = {"L2/3+L4": 13, "L5": 31, "L6": 56}
def laminar(lab, modmask):
    d = pd.DataFrame(dict(layer=U.layer, lab=lab))[modmask]
    out = {}
    for gname, keyf in [("grouped", lambda l: grp.get(l, "none")), ("separate", lambda l: l)]:
        g = d.assign(g=d.layer.map(keyf))
        f = g.groupby("g").lab.apply(lambda x: 100 * np.mean(x == "Fast"))
        s = g.groupby("g").lab.apply(lambda x: 100 * np.mean(np.isin(x, ["SlowPos", "SlowNeg"])))
        out[gname] = pd.DataFrame(dict(fast_pct=f, slow_pct=s, n=g.groupby("g").size()))
    return out
for key, lab, mm in [(k, v, mod) for k, v in cls.items()] + [("variant: recomputed set (a), " + k, v, mod_a) for k, v in cls_a.items() if k.startswith("a ")]:
    o = laminar(lab, mm)
    gr = o["grouped"].loc[["L2/3+L4", "L5", "L6"]]
    win = sum(abs(gr.fast_pct[k] - expF[k]) <= 0.1 * expF[k] for k in expF) + sum(abs(gr.slow_pct[k] - expS[k]) <= 0.1 * expS[k] for k in expS)
    grad = gr.fast_pct.is_monotonic_decreasing and gr.slow_pct.is_monotonic_increasing
    print("\n", key, "| values within tolerance: %d of 6 | gradient holds: %s" % (win, grad))
    print(gr.round(1).T.to_string())
    print(o["separate"].loc[["L2/3", "L4"]].round(1).T.to_string())

**Result: reproduced.** Under reading (a) on the deposited modulated set, Fast units are 74.0% of modulated units in L2/3 and L4, 49.6% in L5 and 23.2% in L6, and Slow units 13.0%, 31.9 to 32.6% and 56.5%: all 6 values within tolerance, and the gradient holds. Separately, L2/3 is 75.0% Fast and L4 73.6%. On the recomputed set 4 of 6 are within tolerance (L6 Fast 26.5%, L2/3 and L4 Slow 18.3%).

## Response classes (Figures 7 and 8)

In [ ]:
U["cls"] = lab_main
U["cls_dep"] = depc
U["cls_a"] = cls_a["a all modulated, 55 ms Medium"]
print("main classes equal the deposited labels for every unit:", bool((U.cls == U.cls_dep).all()))
U["best"] = (R2.argmax(1) + 1) * 5
KINDS = ["F", "M", "S"]

def net_evoked(u, kinds=KINDS):
    """Mean over impulses of each kind (all sequences) of the rate 5-35 ms after onset minus 0-10 ms before."""
    segs = ssc4.impulse_segments(D, u)
    out = {}
    for k in kinds:
        X = np.concatenate([g["X"] for g in segs if g["kind"] == k])
        out[k] = X[:, 15:45].mean() * 1000 - X[:, 0:10].mean() * 1000
    return out

def seq_rates(u, base, sel=None):
    """Per sequence, mean rate 0-150 ms after onset minus the pre-stimulus baseline (base: '25' or '10')."""
    M, t, _ = D.unit_trials(u, sel)
    a = -25 if base == "25" else -10
    out = {}
    for s in ["FFF", "FMS", "SMF", "SSS"]:
        m = (t.stim == s).values
        out[s] = ssc4.rate(M[m], 0, 150).mean() - ssc4.rate(M[m], a, 0).mean()
    return out

def pulse_evoked(u, base, sel=None):
    """Per sequence and pulse, rate 5-35 ms after the pulse onset minus the pre-sequence baseline (S4 Fig)."""
    M, t, _ = D.unit_trials(u, sel)
    si = int(U.session[u])
    a = -25 if base == "25" else -10
    out = {}
    for s in ["FFF", "FMS", "SMF", "SSS"]:
        m = (t.stim == s).values
        if m.sum() == 0:
            continue
        b0 = ssc4.rate(M[m], a, 0).mean()
        for p, on in enumerate(D.onsets(si, s)):
            out[(s, p + 1)] = ssc4.rate(M[m], on + 5, on + 35).mean() - b0
    return out

NE = pd.DataFrame({u: net_evoked(u) for u in U.uid[U.cls != "none"]}).T
SR = {b: pd.DataFrame({u: seq_rates(u, b) for u in U.uid[U.cls != "none"]}).T for b in ["25", "10"]}
NE_a = pd.DataFrame({u: net_evoked(u) for u in U.uid[(U.cls_a != "none")]}).T
SR_a = {b: pd.DataFrame({u: seq_rates(u, b) for u in U.uid[U.cls_a != "none"]}).T for b in ["25", "10"]}
print("class sizes (main):", U.cls.value_counts().to_dict(), "| variant, recomputed set:", U.cls_a.value_counts().to_dict())

### fig7c-fast-impulse-velocity

> In Fast units the evoked rate after an impulse was greater for F than S impulses and
> intermediate for M impulses.

In [ ]:
for nm, ne, c in [("this attempt", NE, "cls"), ("variant: recomputed modulated set (a)", NE_a, "cls_a")]:
    f = ne.loc[U.uid[U[c] == "Fast"]]
    m = f.mean()
    t = stats.ttest_rel(f.F, f.S)
    print("%s: n = %d Fast units, net evoked F %.2f, M %.2f, S %.2f Hz (SEM %s); F > M > S: %s; paired t F vs S t = %.2f p = %.2g"
          % (nm, len(f), m.F, m.M, m.S, f.sem().round(2).to_dict(), m.F > m.M > m.S, t.statistic, t.pvalue))
f = NE.loc[U.uid[U.cls == "Fast"]]
print("by rat:"); print(f.groupby(U.rat).mean().round(2))

**Result: reproduced.** Over 160 Fast units the net evoked rate is 9.99 Hz for F, 8.54 for M and 7.77 for S impulses (paired t F against S p = 1.5e-05); the recomputed set gives the same order (9.59, 8.73, 7.42 Hz).

### fig7c-fast-sequence-rate

> Mean firing rate of Fast units over the whole sequence (0 to 150 ms) followed the mean speed of
> the sequence, highest for FFF, lowest for SSS, and intermediate and equal for FMS and SMF.

In [ ]:
for nm, sr, c in [("this attempt", SR, "cls"), ("variant: recomputed modulated set (a)", SR_a, "cls_a")]:
    for b in ["25", "10"]:
        f = sr[b].loc[U.uid[U[c] == "Fast"]]
        m = f.mean()
        t = stats.ttest_rel(f.FMS, f.SMF)
        print("%s, baseline %s ms: n = %d, FFF %.2f, FMS %.2f, SMF %.2f, SSS %.2f Hz; FFF > FMS,SMF > SSS: %s; FMS vs SMF diff %.2f Hz, t = %.2f, p = %.3f"
              % (nm, b, len(f), m.FFF, m.FMS, m.SMF, m.SSS, m.FFF > max(m.FMS, m.SMF) and min(m.FMS, m.SMF) > m.SSS, m.FMS - m.SMF, t.statistic, t.pvalue))
f = SR["25"].loc[U.uid[U.cls == "Fast"]]
print("by rat (baseline 25 ms):"); print(f.groupby(U.rat).mean().round(2))

**Result: reproduced.** The sequence rate of Fast units orders FFF > FMS and SMF > SSS with both baselines, and FMS and SMF do not differ with either (25-ms baseline: -0.88 Hz, p = 0.058; 0 to 10 ms baseline: -0.91 Hz, p = 0.234). On the recomputed set they differ at p = 0.046 with the 25-ms baseline.

### fig7-fff-adaptation

> Fast RS single units adapted modestly in FFF trains, with pulse 2 to pulse 1 ratio 0.80 and
> pulse 3 to pulse 1 ratio 0.70, significant by t test.

Units: Fast RS single units whose response to F impulses is significant by the impulse t test,
with three readings of the test (one-sided and two-sided over all F impulses, and one-sided over
the first impulse of FFF and FMS only, whose 0 to 10 ms baseline is free of an earlier response),
and all Fast RS single units. The paper's 61 is close to all Fast RS single units (62 here), and
every t test selects about half of them, so the paper's test is not the one run here; the
outcome is taken on all Fast RS single units, the selection whose count matches the paper's (an
assumption). A per-unit ratio is defined only when the unit's pulse 1 response is above 1 Hz, so
that near-zero denominators do not enter; units without are counted.


In [ ]:
def f_responsive(u, side):
    """side 'one'/'two': paired t test over all F impulses (one- or two-sided); 'first': one-sided,
    first impulse of FFF and FMS only, whose 0-10 ms baseline precedes any stimulus."""
    segs = ssc4.impulse_segments(D, u, kinds=["F"], pulses=(0,) if side == "first" else (0, 1, 2))
    tt, p2, p1 = ssc4.impulse_ttest(segs)
    return (p2 if side == "two" else p1) < 0.05
def adaptation(units, seq, base, guard=1.0):
    """Per-unit ratios only where the pulse 1 response is above `guard` Hz (1 Hz, so that near-zero
    denominators do not enter)."""
    pe = pd.DataFrame({u: pulse_evoked(u, base) for u in units}).T
    p1, p2, p3 = pe[(seq, 1)], pe[(seq, 2)], pe[(seq, 3)]
    ok = p1 > guard
    out = {}
    for n, pn in [("2/1", p2), ("3/1", p3)]:
        ratio = (pn / p1)[ok]
        out[n] = dict(mean_ratio=ratio.mean(), sem=ratio.sem(), ratio_of_means=pn.mean() / p1.mean(),
                      p_ratio_vs_1=stats.ttest_1samp(ratio, 1).pvalue, p_paired=stats.ttest_rel(pn, p1).pvalue,
                      n=len(units), n_ratio=int(ok.sum()))
    return pd.DataFrame(out).T, pe
fastrs = {c: U.uid[(U[c] == "Fast") & (U.type == "SU_RS")].values for c in ["cls", "cls_a"]}
resp = {(c, side): [u for u in fastrs[c] if f_responsive(u, side)] for c in fastrs for side in ["one", "two", "first"]}
for c in fastrs:
    resp[(c, "all Fast RS")] = list(fastrs[c])
for k, v in resp.items():
    print(k, "Fast RS single units", len(fastrs[k[0]]), "selected", len(v))
ADAPT, AT = {}, []
for (c, side), units in resp.items():
    for base in ["25", "10"]:
        tab, pe = adaptation(units, "FFF", base)
        ADAPT[(c, side, base)] = pe
        AT.append(tab.assign(classes=c, selection=side, baseline=base).reset_index().rename(columns={"index": "ratio"}))
AT = pd.concat(AT)
AT["ratio_of_means_within"] = np.abs(AT.ratio_of_means - AT.ratio.map({"2/1": 0.80, "3/1": 0.70})) <= AT.ratio.map({"2/1": 0.11, "3/1": 0.14})
AT["mean_ratio_within"] = np.abs(AT.mean_ratio - AT.ratio.map({"2/1": 0.80, "3/1": 0.70})) <= AT.ratio.map({"2/1": 0.11, "3/1": 0.14})
print(AT.round(3).to_string(index=False))
ok = AT.groupby(["classes", "selection", "baseline"]).apply(lambda g: pd.Series(dict(
    b_within_and_p_paired=bool(g.ratio_of_means_within.all() and (g.p_paired < 0.05).all()),
    b_within_and_p_ratio=bool(g.ratio_of_means_within.all() and (g.p_ratio_vs_1 < 0.05).all()),
    a_within_and_p_paired=bool(g.mean_ratio_within.all() and (g.p_paired < 0.05).all()),
    a_within_and_p_ratio=bool(g.mean_ratio_within.all() and (g.p_ratio_vs_1 < 0.05).all()))))
print("\nboth ratios within tolerance and both tests p < 0.05, per combination:"); print(ok.to_string())


**Result: reproduced, resting on the choice of units.** The impulse t test selects 29 to 37 of the 62 Fast RS single units (one-sided, two-sided, first impulse only), not 61, and on those no combination of ratio and test meets the criterion (paired p 0.04 to 0.28, with ratios of means within tolerance). All 62 Fast RS single units, the count closest to the paper's 61, give ratios of population means of 0.744 (2/1) and 0.663 (3/1), within tolerance, with paired t tests p = 0.037 and 0.013 (25-ms baseline; 0.779 and 0.709, the same p, with the 0 to 10 ms baseline). The mean of per-unit ratios (pulse 1 above 1 Hz) is outside tolerance for 2/1 with either baseline (0.42 and 0.65). On the recomputed set the 63 Fast RS units give 0.701 and 0.773 with p = 0.039 and 0.141.

### fig7-sss-no-adaptation

> Adaptation was statistically absent in SSS trains (pulse 2/pulse 1 ratio 1.09, pulse 3/pulse 1 ratio 0.86).

In [ ]:
ST = []
for (c, side), units in resp.items():
    for base in ["25", "10"]:
        tab, _ = adaptation(units, "SSS", base)
        ST.append(tab.assign(classes=c, selection=side, baseline=base).reset_index().rename(columns={"index": "ratio"}))
ST = pd.concat(ST)
ST["ratio_of_means_within"] = np.abs(ST.ratio_of_means - ST.ratio.map({"2/1": 1.09, "3/1": 0.86})) <= ST.ratio.map({"2/1": 0.26, "3/1": 0.35})
ST["mean_ratio_within"] = np.abs(ST.mean_ratio - ST.ratio.map({"2/1": 1.09, "3/1": 0.86})) <= ST.ratio.map({"2/1": 0.26, "3/1": 0.35})
print(ST.round(3).to_string(index=False))
print("smallest p of any test: %.3f" % ST[["p_ratio_vs_1", "p_paired"]].min().min())


**Result: reproduced.** On the same 62 Fast RS units, the SSS ratios of population means are 0.892 (2/1) and 0.871 (3/1), within tolerance, and the paired tests are not significant (p = 0.465 and 0.504). The per-unit ratio 2/1 differs from 1 (mean 0.713, p = 0.046 with the 25-ms baseline), outside tolerance, so the claim holds under the ratio of means with the paired test, the combination that gives fig7-fff-adaptation. Over every selection and baseline the ratios of means are 0.76 to 0.91.

### s4-fms-middle-pulse

> The middle M pulse appeared weaker in FMS than in SMF sequences in Fast RS single units, but the
> difference was not significant.

In [ ]:
for (c, side, base), pe in ADAPT.items():
    a, b = pe[("FMS", 2)], pe[("SMF", 2)]
    t = stats.ttest_rel(a, b)
    print("classes %s, selection %s, baseline %s: n = %d, pulse 2 FMS %.2f, SMF %.2f Hz, diff %.2f Hz, paired t = %.2f, p = %.3f"
          % (c, side, base, len(a), a.mean(), b.mean(), a.mean() - b.mean(), t.statistic, t.pvalue))

**Result: reproduced.** In the 62 Fast RS units the pulse 2 response is 1.26 Hz lower in FMS than SMF with the 25-ms baseline (p = 0.150) and 0.91 Hz lower with the 0 to 10 ms one (p = 0.411). Every selection gives FMS lower and p of 0.06 or more; the 63 units of the recomputed set give 1.60 Hz, p = 0.087, close to the paper's p = 0.08.

### fig7de-medium-suppressed

> Medium units were suppressed by F, M and S impulses and did not distinguish impulse identity or
> sequence identity.

In [ ]:
for nm, ne, sr, c in [("this attempt", NE, SR, "cls"), ("variant: recomputed modulated set (a)", NE_a, SR_a, "cls_a")]:
    us = U.uid[U[c] == "Medium"]
    f = ne.loc[us]
    t1 = stats.ttest_rel(f.F, f.S)
    print("%s: n = %d Medium units; net evoked F %.2f, M %.2f, S %.2f Hz; all negative: %s; F vs S t = %.2f p = %.3f"
          % (nm, len(f), f.F.mean(), f.M.mean(), f.S.mean(), (f.mean() < 0).all(), t1.statistic, t1.pvalue))
    print("   one-sample t against 0:", {k: round(stats.ttest_1samp(f[k], 0).pvalue, 4) for k in KINDS})
    for b in ["25", "10"]:
        s = sr[b].loc[us]
        t2 = stats.ttest_rel(s.FFF, s.SSS)
        print("   baseline %s: sequence rates %s; FFF vs SSS diff %.2f t = %.2f p = %.3f" % (b, s.mean().round(2).to_dict(), (s.FFF - s.SSS).mean(), t2.statistic, t2.pvalue))

**Result: reproduced, resting on the unit set.** The 53 Medium units have negative net evoked rates for all three impulses (-0.27 F, -0.39 M, -0.52 Hz S), and they do not distinguish F from S (p = 0.766) or FFF from SSS (p = 0.156 and 0.200), which meets the criterion. None of the three rates differs from 0 (p 0.45 to 0.72), so the suppression is small. On the recomputed set (50 units) the F rate is +0.20 Hz and the criterion is not met.

### fig8a-slow-positive-no-sequence-difference

> Slow Positive units showed small, sustained responses mostly to the second and third impulses,
> evident for F and M but not S impulses, and their mean firing did not differ between FFF, FMS,
> SMF and SSS.

In [ ]:
def rm_anova(Y):
    """One-way repeated-measures ANOVA, Y units x conditions."""
    Y = np.asarray(Y, float)
    n, k = Y.shape
    gm = Y.mean()
    ss_c = n * ((Y.mean(0) - gm) ** 2).sum()
    ss_s = k * ((Y.mean(1) - gm) ** 2).sum()
    ss_e = ((Y - gm) ** 2).sum() - ss_c - ss_s
    F = (ss_c / (k - 1)) / (ss_e / ((n - 1) * (k - 1)))
    return F, stats.f.sf(F, k - 1, (n - 1) * (k - 1))
for nm, ne, sr, c in [("this attempt", NE, SR, "cls"), ("variant: recomputed modulated set (a)", NE_a, SR_a, "cls_a")]:
    us = U.uid[U[c] == "SlowPos"]
    f = ne.loc[us]
    tt = {k: stats.ttest_1samp(f[k], 0) for k in KINDS}
    print("%s: n = %d Slow Positive; net evoked %s Hz; one-sample t p: %s" % (nm, len(f), f.mean().round(2).to_dict(),
          {k: "%.3g (t %.2f)" % (v.pvalue, v.statistic) for k, v in tt.items()}))
    for b in ["25", "10"]:
        s = sr[b].loc[us]
        F, p = rm_anova(s.values)
        print("   baseline %s: sequence rates %s; RM-ANOVA F = %.2f p = %.3f" % (b, s.mean().round(2).to_dict(), F, p))
pp = pd.DataFrame({u: pulse_evoked(u, "25") for u in U.uid[U.cls == "SlowPos"]}).T
print("pulse-evoked rate above the 25-ms baseline by pulse (mean over Slow Positive units and sequences):",
      {p: round(pp[[c for c in pp.columns if c[1] == p]].mean(axis=1).mean(), 2) for p in (1, 2, 3)})

**Result: partially reproduced.** Slow Positive units respond to F (1.77 Hz, p = 0.0009) and M impulses (2.71 Hz, p = 0.0011), but also to S (1.30 Hz, p = 0.021), which the claim says they do not. Their sequence rates do not differ with the 0 to 10 ms baseline (repeated-measures ANOVA p = 0.329) and differ with the 25-ms baseline (p = 0.005). Responses grow over the pulses (-0.43, 1.87, 5.80 Hz for pulses 1 to 3). The recomputed set gives the same for the S response (p = 0.022).

### fig8b-slow-negative-no-impulse-response

> Slow Negative units did not respond to impulses, and their firing declined steadily over time.

In [ ]:
for nm, ne, c in [("this attempt", NE, "cls"), ("variant: recomputed modulated set (a)", NE_a, "cls_a")]:
    us = U.uid[U[c] == "SlowNeg"]
    f = ne.loc[us]
    tt = {k: stats.ttest_1samp(f[k], 0) for k in KINDS}
    pos = [k for k, v in tt.items() if v.statistic > 0 and v.pvalue < 0.05]
    pop = np.mean([unit_psth(u, 0, 180) for u in us], 0)
    sl = stats.linregress(np.arange(180), pop)
    print("%s: n = %d Slow Negative; net evoked %s Hz; t (p): %s; significantly positive kinds: %s; PSTH slope 0-180 ms %.2f Hz/100 ms (p = %.2g)"
          % (nm, len(f), f.mean().round(2).to_dict(), {k: "%.2f (%.3g)" % (v.statistic, v.pvalue) for k, v in tt.items()}, pos, 100 * sl.slope, sl.pvalue))

**Result: reproduced.** No impulse kind evokes a significant increase in the 46 Slow Negative units (t 0.78, -0.77 and 0.94), and their population rate falls from 0 to 180 ms (-5.90 Hz per 100 ms).

### fig8d-slow-positive-right-choice-l5

> Slow Positive units fired more on right-choice trials (contralateral to the recording) for all
> stimuli; right-choice bias, measured 5 to 50 ms after the start of the final impulse, was
> significant in L5a and L5b but not in other layers.

In [ ]:
def choice_bias(u):
    """Rate 5-50 ms after the third impulse onset, right minus left: (a) all trials pooled, (b) per sequence, averaged."""
    M, t, _ = D.unit_trials(u)
    si = int(U.session[u])
    r = np.zeros(len(t))
    for s in ssc4.STIMS:
        m = (t.stim == s).values
        on = D.onsets(si, s)[2]
        r[m] = ssc4.rate(M[m], on + 5, on + 50)
    right = t.chosen.values == 1
    a = (r[right].mean(), r[~right].mean())
    d = []
    for s in ssc4.STIMS:
        m = (t.stim == s).values
        if (m & right).any() and (m & ~right).any():
            d.append((r[m & right].mean(), r[m & ~right].mean()))
    b = tuple(np.mean(d, 0))
    return a + b
def bias_table(cl, c="cls"):
    us = U.uid[U[c].isin(cl)]
    cb = pd.DataFrame({u: choice_bias(u) for u in us}, index=["aR", "aL", "bR", "bL"]).T
    cb["layer"] = U.layer[us].values
    rows = []
    for lay in order:
        g = cb[cb.layer == lay]
        row = dict(layer=lay, n=len(g))
        for rd in "ab":
            if len(g) > 1:
                t = stats.ttest_rel(g[rd + "R"], g[rd + "L"])
                row.update({rd + "_diff": (g[rd + "R"] - g[rd + "L"]).mean(), rd + "_t": t.statistic, rd + "_p": t.pvalue})
        rows.append(row)
    return pd.DataFrame(rows).set_index("layer")
for nm, c in [("this attempt", "cls"), ("variant: recomputed modulated set (a)", "cls_a")]:
    bt8 = bias_table(["SlowPos"], c)
    print(nm, "Slow Positive"); print(bt8.round(4).to_string())
    for rd in "ab":
        ok = (bt8.loc[["L5a", "L5b"], rd + "_p"] < 0.05).all() and (bt8.loc[["L5a", "L5b"], rd + "_diff"] > 0).all() and \
             not (bt8.loc[["L2/3", "L4", "L6"], rd + "_p"] < 0.05).any()
        print("  reading %s: L5a and L5b significant with right higher, other layers not: %s" % (rd, ok))
print("\nVariant (not judged): Fast and Medium units by layer")
print(bias_table(["Fast", "Medium"]).round(4).to_string())
print("paper's starred p-values: 0.022, 9.5e-4, 7.5e-5")

**Result: reproduced, resting on the unit set.** Under reading (a), Slow Positive units in L5a (7 units, 4.58 Hz, p = 0.049) and L5b (14 units, 6.62 Hz, p = 0.0051) fire more on right-choice trials, and L4 and L6 show no bias (L2/3 has one unit, so no test). Under reading (b) L5a is not significant (p = 0.074). On the recomputed set L5a is not significant under either reading (6 units, p = 0.117 and 0.121). The printed p-values here are not the paper's (0.022, 9.5e-4, 7.5e-5).

### fig8e-choice-divergence-125ms

> In L5 Slow Positive units, firing on right- and left-choice trials diverged after the second
> impulse and was consistently significant by 125 ms, before the earliest withdrawals (150 ms) and
> the mean withdrawal time (190 ms).

Per unit the PSTH for each choice is the mean of its per-sequence PSTHs (sequences with trials of
that choice). "Lasts to the end of the third impulse" is taken as lasting to 150 ms, the end of
the 0-150 ms sequence window of the paper, a standard choice; 140 and 160 ms are shown too. A
trailing 10-ms window (value at t the mean of t-9 to t) is a variant of reading (a).

In [ ]:
def choice_psth(u, a=-50, b=250):
    M, t, _ = D.unit_trials(u)
    out = {}
    for c in (1, 0):
        ps = [M[((t.stim == s) & (t.chosen == c)).values, a - ssc4.T0:b - ssc4.T0].mean(0) * 1000
              for s in ssc4.STIMS if ((t.stim == s) & (t.chosen == c)).any()]
        out[c] = np.mean(ps, 0)
    return out
TAX = np.arange(-50, 250)
def sliding(us, reading):
    R = np.array([choice_psth(u)[1] for u in us])
    L = np.array([choice_psth(u)[0] for u in us])
    if reading == "a":    # 10-ms windows stepped by 1 ms, centred
        k = np.ones(10) / 10
        R = np.array([np.convolve(r, k, "same") for r in R]); L = np.array([np.convolve(r, k, "same") for r in L])
        t = TAX
    elif reading == "a_trailing":    # 10-ms windows stepped by 1 ms, value at t = mean of t-9..t
        k = np.ones(10) / 10
        R = np.array([np.convolve(r, k)[:len(r)] for r in R]); L = np.array([np.convolve(r, k)[:len(r)] for r in L])
        t = TAX
    else:                 # 5-ms bins
        R = R.reshape(len(us), -1, 5).mean(2); L = L.reshape(len(us), -1, 5).mean(2)
        t = TAX[::5]
    tt = stats.ttest_rel(R, L, axis=0)
    return t, tt.pvalue, (R - L).mean(0)
def onset_of_run(t, p, end):
    s = p < 0.05
    m = t <= end
    for i in np.where(m & s)[0]:
        if s[i:np.searchsorted(t, end, "right")].all():
            return t[i]
    return np.nan
for nm, c in [("this attempt", "cls"), ("variant: recomputed modulated set (a)", "cls_a")]:
    us = U.uid[(U[c] == "SlowPos") & U.layer.isin(["L5a", "L5b"])].values
    for rd in ["a", "b", "a_trailing"]:
        t, p, d = sliding(us, rd)
        ons = {end: onset_of_run(t, p, end) for end in (140, 150, 160)}
        first = t[(p < 0.05) & (t >= 0)]
        print("%s, reading %s: n = %d L5 Slow Positive; onset of the run lasting to 140/150/160 ms: %s; first significant time after 0: %s"
              % (nm, rd, len(us), ons, first[0] if len(first) else None))
us = U.uid[(U.cls == "SlowPos") & U.layer.isin(["L5a", "L5b"])].values
tr = pd.concat([D.unit_trials(u)[1].assign(session=int(U.session[u])) for u in us]).drop_duplicates(["session", "trial"])
print("withdrawal after onset over the %d trials of these units' sessions: min %.0f, 1st percentile %.1f, mean %.1f ms"
      % (len(tr), tr.withdraw_t.min(), np.percentile(tr.withdraw_t, 1), tr.withdraw_t.mean()))
t, p, d = sliding(us, "a")
print("right minus left (Hz), 10-ms windows, at 50/75/100/125/150 ms:", {x: round(float(d[TAX == x][0]), 2) for x in (50, 75, 100, 125, 150)})

**Result: reproduced.** In 21 L5 Slow Positive units the right-left difference is significant without a break from 120 ms to the end of the sequence with 10-ms windows (reading a), whether the end is taken as 140, 150 or 160 ms; a trailing window gives 124 ms. With 5-ms bins the run starts at 120 ms and lasts to 150 ms. Withdrawals over the same sessions are at 154 ms at the earliest and 193.3 ms on average, all three values within tolerance.

### fig8d-fast-medium-no-choice-bias

> Choice-related activity was absent in L4 Fast units.

In [ ]:
for nm, c in [("this attempt", "cls"), ("variant: recomputed modulated set (a)", "cls_a")]:
    us = U.uid[(U[c] == "Fast") & (U.layer == "L4")].values
    for rd in "ab":
        t, p, d = sliding(us, rd)
        m = (t >= 0) & (t <= 150) if rd == "a" else (t >= 0) & (t + 5 <= 150)
        sigt = t[m & (p < 0.05)]
        print("%s, reading %s: n = %d L4 Fast; significant times 0-150 ms: %d (%s); run lasting to 150 ms from: %s; largest |R-L| 0-150 ms %.2f Hz"
              % (nm, rd, len(us), len(sigt), sigt[:20].tolist(), onset_of_run(t, p, 150), np.abs(d[m]).max()))
print("\nVariant: Fast and Medium right-choice bias by layer is printed under fig8d-slow-positive-right-choice-l5.")

**Result: reproduced.** With 5-ms bins (reading b) no bin from 0 to 150 ms differs between right- and left-choice trials in the 53 L4 Fast units. With 1-ms steps of 10-ms windows (reading a), 5 steps (28 to 32 ms and 91 ms) reach p < 0.05, with no run lasting to the end of the sequence. The largest difference is 2.4 to 3.3 Hz.

### s5-slow-positive-no-integration

> Within right- or left-choice trials, L5 Slow Positive units fired little to pulses 1 and 2, and
> their pulse 3 firing was the same for FFF, FMS, SMF and SSS and did not correlate with mean
> speed of the sequence or of the last two impulses.

Impulse mean speed is taken from the S1 Fig speed profiles in S2 Data (sheet "Fig S1"), as the
criterion says: each impulse is the span where the panel is more than 0.2 mm from its rest
position, and its mean speed the mean of the speed column over that span, per sequence. The
first run used a ramp on the Table 1 amplitudes, now a variant.

In [ ]:
s1f = S2["Fig S1"].iloc[3:].astype(float).values
tt_ms = s1f[:, 0] * 1000
inst_speed = {}
for j, s in enumerate(["FFF", "FMS", "SMF", "SSS"]):
    pos, sp = s1f[:, 1 + j], s1f[:, 5 + j]
    on = (pos - np.median(pos[tt_ms < 60])) > 0.2      # panel more than 0.2 mm out of its rest position
    d = np.diff(np.r_[0, on.astype(int), 0])
    seg = [(a, b) for a, b in zip(np.where(d == 1)[0], np.where(d == -1)[0]) if b - a > 12]
    inst_speed[s] = [sp[a:b].mean() for a, b in seg]
    print("%s impulses from S2 Data 'Fig S1' (onset ms, duration ms, mean speed mm/s):" % s,
          [(round(tt_ms[a], 1), round(tt_ms[b - 1] - tt_ms[a], 1), round(sp[a:b].mean(), 1)) for a, b in seg])
assert all(len(v) == 3 for v in inst_speed.values())
ramp = {"F": 2 * 1.03 / 16 * 1000, "M": 2 * 1.15 / 22 * 1000, "S": 2 * 1.14 / 28 * 1000}
print("Table 1 ramp speeds (first run, now a variant):", {k: round(v, 1) for k, v in ramp.items()})
for nm, c in [("this attempt", "cls"), ("variant: recomputed modulated set (a)", "cls_a")]:
    us = U.uid[(U[c] == "SlowPos") & U.layer.isin(["L5a", "L5b"])].values
    for ch, chn in [(1, "right"), (0, "left")]:
        for base in ["25", "10"]:
            pe = pd.DataFrame({u: pulse_evoked(u, base, lambda t, ch=ch: (t.complete & (t.chosen == ch)).values) for u in us}).T
            p3 = pe[[(s, 3) for s in ["FFF", "FMS", "SMF", "SSS"]]].dropna()
            F, p = rm_anova(p3.values)
            pop = p3.mean()
            seqs = ["FFF", "FMS", "SMF", "SSS"]
            r1 = stats.pearsonr([ssc4.MEAN_SPEED[s] for s in seqs], pop.values)
            r2 = stats.pearsonr([np.mean(inst_speed[s][1:3]) for s in seqs], pop.values)
            r3 = stats.pearsonr([np.mean(inst_speed[s][0:2]) for s in seqs], pop.values)
            r2r = stats.pearsonr([(ramp[s[1]] + ramp[s[2]]) / 2 for s in seqs], pop.values)
            p12 = pe[[c2 for c2 in pe.columns if c2[1] in (1, 2)]].mean().mean()
            print("%s, %s choice, baseline %s: n = %d; pulse 1-2 mean %.2f Hz; pulse 3 %s; RM-ANOVA F = %.2f p = %.3f; r(seq speed) = %.2f p = %.2f; r(pulses 2,3) = %.2f p = %.2f; r(pulses 1,2) = %.2f p = %.2f; variant, r(pulses 2,3, Table 1 ramp) = %.2f p = %.2f"
                  % (nm, chn, base, len(p3), p12, pop.round(2).to_dict(), F, p, r1[0], r1[1], r2[0], r2[1], r3[0], r3[1], r2r[0], r2r[1]))

**Result: partially reproduced.** L5 Slow Positive units fire little to pulses 1 and 2 (2.00 Hz on right-choice and 0.39 Hz on left-choice trials, 25-ms baseline) against 5.3 to 12.6 Hz for pulse 3 on right-choice trials, and pulse 3 does not correlate with sequence speed or with the S1 Fig speed of pulses 2 and 3 or 1 and 2 with the 25-ms baseline (all p of 0.17 or more). But pulse 3 differs across sequences on right-choice trials (ANOVA p = 0.016; SMF 5.32 Hz against 11.8 to 12.6), and with the 0 to 10 ms baseline the ANOVA is not significant but the left-choice correlation with pulses 2 and 3 is (r = -0.96, p = 0.04). So no baseline meets the criterion. On the recomputed set the 25-ms baseline meets it (right-choice ANOVA p = 0.052).

### s6-head-position-choice (S2 Data, S6 Fig)

> In high-speed video of 43 trials, head angle and whisker tip trajectories did not differ between
> right- and left-choice trials, while head position differed modestly from 100 ms, by 0.6 mm at
> 125 ms.

S2 Data gives the mean and SEM per frame for 22 left- and 21 right-choice trials, so the t tests
are computed from those summaries.

In [ ]:
s6 = S2["Fig S6"]
cols = s6.iloc[3, :].tolist()
left = pd.DataFrame(s6.iloc[4:23].values.astype(float), columns=cols)
right = pd.DataFrame(s6.iloc[26:45].values.astype(float), columns=cols)
nL, nR = 22, 21
out = {"time": left["Time (ms)"].values}
for v in ["Head Angle", "W Lat", "W RC", "Head dx", "Head dy"]:
    mL, sL = left[[c for c in cols if c.startswith(v) and "mean" in c][0]], left[[c for c in cols if c.startswith(v) and "sem" in c.replace("  ", " ")][0]]
    mR, sR = right[[c for c in cols if c.startswith(v) and "mean" in c][0]], right[[c for c in cols if c.startswith(v) and "sem" in c.replace("  ", " ")][0]]
    sdL, sdR = sL * np.sqrt(nL), sR * np.sqrt(nR)
    w = stats.ttest_ind_from_stats(mR, sdR, nR, mL, sdL, nL, equal_var=False)
    s = stats.ttest_ind_from_stats(mR, sdR, nR, mL, sdL, nL, equal_var=True)
    out[v + " R-L"] = (mR - mL).values
    out[v + " p_welch"] = w.pvalue
    out[v + " p_student"] = s.pvalue
S6 = pd.DataFrame(out)
print(S6.round(3).to_string())
for v in ["Head Angle", "W Lat", "W RC", "Head dx", "Head dy"]:
    for k in ["p_welch", "p_student"]:
        sigt = S6.time[S6[v + " " + k] < 0.05].tolist()
        print("%s %s: frames with p < 0.05: %s" % (v, k, sigt))
i = int(np.argmin(np.abs(S6.time - 125)))
print("at %.1f ms: head dx right minus left %.3f mm, head dy %.3f mm" % (S6.time[i], S6["Head dx R-L"][i], S6["Head dy R-L"][i]))

**Result: reproduced.** From S2 Data, head position along x differs between right- and left-choice trials at every frame from 100.8 ms (p = 0.037) on, by 0.595 mm at 126 ms. Head angle, both whisker tip coordinates and head y differ at no frame.

### s7-no-fs-l5-slow-positive

> All four response classes (Fast, Medium, Slow Positive, Slow Negative) included RS single units,
> FS single units and multi-unit clusters; few FS units were in the Slow classes and none among L5
> Slow Positive units.

In [ ]:
for nm, c in [("this attempt", "cls"), ("variant: recomputed modulated set (a)", "cls_a")]:
    ct = pd.crosstab(U[c], U.type).drop("none")
    ct["FS_frac"] = ct.SU_FS / ct[["MU", "SU_FS", "SU_RS"]].sum(axis=1)
    print(nm); print(ct.round(3))
    l5sp = U[(U[c] == "SlowPos") & U.layer.isin(["L5a", "L5b"])].type.value_counts().to_dict()
    print("  L5 Slow Positive by type:", l5sp)
    print("  every class holds all three types:", bool((ct[["MU", "SU_FS", "SU_RS"]] > 0).all().all()),
          "| FS fraction of each Slow class below Fast:", bool((ct.loc[["SlowPos", "SlowNeg"], "FS_frac"] < ct.loc["Fast", "FS_frac"]).all()))

**Result: reproduced.** Every class holds RS, FS and multi-units; the FS fraction is 0.137 in Slow Positive and 0.043 in Slow Negative against 0.194 in Fast; the 21 L5 Slow Positive units are 14 RS and 7 multi-units, with no FS unit.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| fig2ab-fms-smf-not-discriminated | partially reproduced (5 of 8 rats; 62SC not in the data) |
| training-days | partially reproduced (minimum 11 or 12 days, not 8) |
| fig2c-choice-follows-mean-speed | reproduced |
| fig2d-fixed-panel-impairs | reproduced (S2 Data) |
| dprime-fff-sss-range | partially reproduced (highest 1.09, not 1.5) |
| fig3-fsfs-sffs-intensity-only | reproduced (S1 Data) |
| units-recorded | reproduced (but 506 units against 473) |
| baseline-rates | partially reproduced (RS maximum 11.4 Hz) |
| temporal-modulation-count | reproduced |
| modulated-higher-baseline | reproduced |
| fig4d-evoked-modulation-by-layer | partially reproduced (4 of 5 layers) |
| latency-by-layer | partially reproduced (ordering only, two-sided test) |
| fig4d-gradual-decrease | not reproduced (L5a also decreases) |
| fig6-regression-significant | reproduced |
| fig6e-unit-classes | reproduced |
| fig6fg-laminar-classes | reproduced |
| fig7c-fast-impulse-velocity | reproduced |
| fig7c-fast-sequence-rate | reproduced |
| fig7-fff-adaptation | reproduced (on all 62 Fast RS units, an assumption) |
| fig7-sss-no-adaptation | reproduced |
| s4-fms-middle-pulse | reproduced |
| fig7de-medium-suppressed | reproduced (on the deposited set, an assumption; suppression not significant) |
| fig8a-slow-positive-no-sequence-difference | partially reproduced (S response too) |
| fig8b-slow-negative-no-impulse-response | reproduced |
| fig8d-slow-positive-right-choice-l5 | reproduced (reading a, on the deposited set, an assumption) |
| fig8d-fast-medium-no-choice-bias | reproduced |
| fig8e-choice-divergence-125ms | reproduced |
| s5-slow-positive-no-integration | partially reproduced (pulse 3 differs across sequences) |
| s6-head-position-choice | reproduced (S2 Data) |
| s7-no-fs-l5-slow-positive | reproduced |

The decoder claims of Figure 9 are in `decoders.ipynb`.

How the analysis differs from the paper's: behaviour rests on the five recorded rats, not
eight; units are all 506 deposited, not the paper's 473, since the columns to drop cannot be
identified; the temporally modulated units are those the data set marks (its stored p-values,
which the descriptor names as the paper's units), as the Methods' test recomputed here gives the
same count but a different set (73% agreement); the classes are recomputed from the regression
and equal the data set's labels; the impulse t test selects about half of the Fast RS units, so
the adaptation claims rest on all 62 Fast RS units, close to the paper's 61; impulse speeds for
s5 come from the S1 Fig profiles in S2 Data.

What was found about the data set: the column map (`MapPosition`) the descriptor documents is
missing, so 33 units beyond the paper's 473 cannot be removed (by S1 Table they are in L2/3 to
L5b, none in L6); the stored temporal-modulation p-values use a -50 to 300 ms window, not the
Methods' window, and neither that window nor a bin-shuffle null reproduces them; the fixed-panel
sessions and three behaviour-only rats are not deposited; the stored regression inputs show that
completed trials were analysed; no recording is deposited twice; spikes and trials share one
clock.
